In [2]:
import argparse
import copy
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from dataset.CremadDataset import CremadDataset
from models.basic_model import AClassifier,VClassifier
from utils.utils import setup_seed, weight_init
import time
import matplotlib.pyplot as plt

In [3]:
def get_arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', default='CREMAD', type=str,
                        help='VGGSound, KineticSound, CREMAD, AVE')
    parser.add_argument('--batch_size', default=64, type=int)
    parser.add_argument('--epochs', default=150, type=int)
    parser.add_argument('--embed_dim', default=512, type=int)
    parser.add_argument('--optimizer', default='SGD', type=str)
    parser.add_argument('--learning_rate', default=0.001, type=float, help='initial learning rate')
    parser.add_argument('--ckpt_path', default='ckpt', type=str, help='path to save trained models')
    parser.add_argument('--train', action='store_true', help='turn on train mode')
    parser.add_argument('--random_seed', default=0, type=int)
    parser.add_argument('--gpu', type=int, default=0)  # gpu
    parser.add_argument('--no_cuda', action='store_true', help='Disable CUDA')
    return parser.parse_args(args=[])

In [4]:
args = get_arguments()

In [5]:
train_dataset = CremadDataset(args, mode='train')
test_dataset = CremadDataset(args, mode='test')

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=64,shuffle=True,num_workers=10)
test_dataloader = DataLoader(test_dataset, batch_size=64,num_workers=10)

In [7]:
device = torch.device('cuda')

In [17]:
model1 = AClassifier(args)
model2 = VClassifier(args)
model1.to(device)
model2.to(device)
model1.load_state_dict(torch.load('../CKD-CREMAD/ckpt/unimodality-audio/model-CREMAD-frm1-bsz64-lr0.01/best.pt')['model'])
model2.load_state_dict(torch.load('../CKD-CREMAD/ckpt/unimodality-visual/model-CREMAD-frm1-bsz64-lr0.01/best.pt')['model']) 

/tmp/ipykernel_179/2460782569.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1.load_state_dict(torch.load('../CKD-CREMAD/ckpt/unimodality-audio/model-CREMAD-frm1-b

<All keys matched successfully>

In [10]:
class FDFilter(nn.Module):
    def __init__(self):
        super(FDFilter, self).__init__()
        self.filter1 = nn.Parameter(torch.zeros(257), requires_grad=False)
        self.filter2 = nn.Parameter(torch.zeros(257), requires_grad=False)
        self.init_filter()

    def init_filter(self):
        self.filter1.data[:129] = 1
        self.filter2.data[129:] = 1
        
    def forward(self, x):
        freq_domain = torch.fft.rfft(x, dim=1)

        binary_filter1 = (self.filter1 >= 0.5).float()
        filtered_freq_domain1 = freq_domain * binary_filter1
        binary_filter2 = (self.filter2 >= 0.5).float()
        filtered_freq_domain2 = freq_domain * binary_filter2

        low_freq_features1 = torch.fft.irfft(filtered_freq_domain1, dim=1).real
        low_freq_features2 = torch.fft.irfft(filtered_freq_domain2, dim=1).real
        return low_freq_features1,low_freq_features2

In [16]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm

def calculate_cross_modal_similarity(dataloader, model1, model2, fdfilter, device):
    # Ensure all models are in evaluation mode
    model1.eval()
    model2.eval()
    fdfilter.eval()

    # Initialize lists to store similarity scores for each sample
    low_freq_sims_list = []
    high_freq_sims_list = []
    freq_sims_list = []

    # Disable gradient calculation for efficiency
    with torch.no_grad():
        # Use tqdm to show processing progress
        for spec, image, label in tqdm(dataloader, desc="Calculating cross-modal similarity"):
            # Move data to the specified device
            spec = spec.to(device)
            image = image.to(device)
            B = label.shape[0] # Batch size

            # 1. Extract raw features
            a, _ = model1(spec.unsqueeze(1).float())
            v, _ = model2(image.float(), B)

            # 2. Extract high-frequency and low-frequency features
            a_low, a_high = fdfilter(a)
            v_low, v_high = fdfilter(v)

            # 3. Calculate cross-modal cosine similarity
            sim_low = F.cosine_similarity(a_low, v_low, dim=1)
            sim_high = F.cosine_similarity(a_high, v_high, dim=1)
            sim = F.cosine_similarity(a, v, dim=1)

            # 4. Store current batch's scores into lists
            #    Use .cpu() to move data back to CPU, preventing GPU memory accumulation
            low_freq_sims_list.append(sim_low.cpu())
            high_freq_sims_list.append(sim_high.cpu())
            freq_sims_list.append(sim.cpu())

    # 5. Concatenate all batch scores from the lists into one large 1D tensor
    all_low_freq_sims = torch.cat(low_freq_sims_list, dim=0)
    all_high_freq_sims = torch.cat(high_freq_sims_list, dim=0)
    all_freq_sims = torch.cat(freq_sims_list, dim=0)

    # 6. Calculate the average similarity across all samples
    avg_sim_low = torch.mean(all_low_freq_sims).item()
    avg_sim_high = torch.mean(all_high_freq_sims).item()
    avg_sim = torch.mean(all_freq_sims).item()

    return avg_sim_low, avg_sim_high, avg_sim

print("Starting final validation: Calculating cross-modal similarity...")

fdfilter = FDFilter().cuda()
# Call the function for calculation
# Note: This function can handle any batch_size, not necessarily 1
avg_low_freq_cross_sim, avg_high_freq_cross_sim, avg_freq_cross_sim = calculate_cross_modal_similarity(
    test_dataloader, model1, model2, fdfilter, device
)

# --- Print the final, decisive results ---
print("\n" + "="*50)
print("Cross-Modal Similarity Results")
print("="*50)
print(f"Average Low-Frequency Cross-Modal Similarity: {avg_low_freq_cross_sim:.4f}")
print(f"Average High-Frequency Cross-Modal Similarity: {avg_high_freq_cross_sim:.4f}")
print(f"Average Raw Feature Cross-Modal Similarity: {avg_freq_cross_sim:.4f}")

print("="*50)


Starting final validation: Calculating cross-modal similarity...


Calculating cross-modal similarity: 100%|██████████| 12/12 [00:01<00:00,  6.26it/s]


Cross-Modal Similarity Results
Average Low-Frequency Cross-Modal Similarity: 0.9113
Average High-Frequency Cross-Modal Similarity: -0.0234
Average Raw Feature Cross-Modal Similarity: 0.8379
